# Fine-tune PP-OCRv5 mobile rec trên crop biển số Việt Nam

**Mục tiêu:** nâng độ chính xác chuỗi **biển 2 dòng** (hiện ~0,60 trên tập đánh giá 2.801 mẫu)
bằng cách fine-tune bộ nhận dạng ký tự trên đúng phân phối mà hệ thống thật đưa vào OCR
(strip 2-dòng-ghép-ngang, cao 64 px).

**Cần có trước:** `rec_finetune.zip` (~40 MB) đã nằm ở `MyDrive/DATN/` của **tài khoản Google đang
chạy notebook này**. Sinh lại bằng: `backend/.venv/Scripts/python scripts/dataset/build_rec_finetune_set.py`
rồi nén thư mục `datasets/processed/rec_finetune`.

**Runtime:** Thời gian chạy → Thay đổi loại thời gian chạy → **T4 GPU** → Lưu.

---
### Cách chạy
Chạy **tuần tự từng cell**, mỗi cell có dòng `MOC:` ghi thứ phải nhìn thấy trước khi sang cell sau.
Cell nào tải mạng đều có `--retries` hoặc `wget -c`, gặp lỗi mạng cứ chạy lại chính cell đó.

> ⚠️ **Hai lỗi đã gặp thật, notebook này đã vá sẵn — đừng sửa lại theo tài liệu cũ trên mạng:**
> 1. `paddlepaddle-gpu==3.3.1` **chỉ có trên index của Paddle**; thiếu `-i …` là pip tìm PyPI và báo
>    *"No matching distribution"* (PyPI dừng ở 2.6.2).
> 2. Bộ trọng số pretrained nằm ở **paddle-model-ecology**, không phải `paddleocr.bj.bcebos.com`.
>    Link cũ trả về JSON `NoSuchKey` 117 byte — tải xong vẫn 'thành công' nhưng train sẽ hỏng.


In [ ]:
# 1) Kiem tra GPU
#    MOC: phai in ra 'GPU 0: Tesla T4 ...'. Neu khong co: doi runtime sang T4 roi chay lai.
!nvidia-smi -L


In [ ]:
# 2) Cai PaddlePaddle GPU 3.3.1 (dung phien ban voi he thong local)
#    Wheel ~2 GB, CDN Baidu hay cham: cell nay co the mat 5-20 phut. Cu de no chay.
#    MOC: dong cuoi in 'CUDA: True'.
#    (Cac dong ERROR ve torch/nvidia-cudnn la canh bao phu thuoc cua pip — VO HAI.)
!pip install -q --timeout 300 --retries 8 paddlepaddle-gpu==3.3.1 \
    -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
import paddle; print('CUDA:', paddle.device.is_compiled_with_cuda())


In [ ]:
# 2b) DU PHONG — chi chay neu cell 2 timeout mai khong xong.
#     wget -c noi tiep phan da tai: dut mang thi CHAY LAI CHINH CELL NAY, khong mat tu dau.
import re, urllib.request
idx = 'https://www.paddlepaddle.org.cn/packages/stable/cu126/paddlepaddle-gpu/'
html = urllib.request.urlopen(idx).read().decode()
name = re.search(r'paddlepaddle_gpu-3\.3\.1[^"]*cp312[^"]*linux_x86_64\.whl', html).group(0)
url = f'https://paddle-whl.bj.bcebos.com/stable/cu126/paddlepaddle-gpu/{name}'
print(url)
!wget -c -q --show-progress --tries=10 --timeout=60 {url}
!pip install -q paddlepaddle_gpu-3.3.1*.whl
import paddle; print('CUDA:', paddle.device.is_compiled_with_cuda())


In [ ]:
# 3) Lay ma nguon PaddleOCR (chua tools/train.py) + requirements
#    MOC: in 'PaddleOCR san sang' (khoang 15-60 giay).
%cd /content
!rm -rf PaddleOCR
!git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git
%cd /content/PaddleOCR
!pip install -q --timeout 180 --retries 5 -r requirements.txt
print('PaddleOCR san sang')


In [ ]:
# 4) Mount Drive va giai nen dataset
#    Colab se hoi quyen truy cap Drive -> bam 'Connect to Google Drive' va cho phep.
#    MOC: in dung '6672 ... train.txt' va '571 ... val.txt'.
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/data && mkdir -p /content/data
!unzip -q /content/drive/MyDrive/DATN/rec_finetune.zip -d /content/data
!wc -l /content/data/rec_finetune/train.txt /content/data/rec_finetune/val.txt
!head -2 /content/data/rec_finetune/train.txt
!head -3 /content/data/rec_finetune/dict36.txt


In [ ]:
# 5) Tai bo trong so pretrained cua en_PP-OCRv5_mobile_rec
#    URL nay lay tu docs cua chinh PaddleOCR (paddle-model-ecology).
#    MOC: file .pdparams khoang 70 MB. Neu chi vai tram BYTE thi do la trang loi JSON,
#         KHONG phai model — xoa di va kiem tra lai URL truoc khi train.
!mkdir -p /content/pretrained
!wget -c -q --show-progress --tries=10 --timeout=60 \
  -O /content/pretrained/en_PP-OCRv5_mobile_rec_pretrained.pdparams \
  https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/en_PP-OCRv5_mobile_rec_pretrained.pdparams
!ls -la /content/pretrained


In [ ]:
# 6) Xac dinh file config (da kiem chung tren ban clone thang 7/2026)
#    MOC: in ra duong dan .yml va dong 'Config OK'.
import os
CONFIG = 'configs/rec/PP-OCRv5/multi_language/en_PP-OCRv5_mobile_rec.yaml'
if not os.path.exists(CONFIG):
    import glob
    found = glob.glob('configs/rec/**/*en_PP-OCRv5_mobile*.y*ml', recursive=True)
    print('Duong dan mac dinh khong con, tim thay:', found)
    CONFIG = found[0]
print('CONFIG =', CONFIG, '| Config OK')


### Về cảnh báo `shape ... not matched` ở đầu quá trình huấn luyện

Khi bắt đầu, log sẽ in vài dòng `WARNING: The shape of model params head.ctc_head.fc.weight
paddle.Size([120, 37]) not matched with loaded params ... paddle.Size([120, 438])`.

**Đây là chủ đích, không phải lỗi.** Model gốc có 438 lớp ký tự (tiếng Anh đầy đủ); đồ án dùng
**charset 36 ký tự** (`0-9A-Z`, quyết định Phase 1) nên hai lớp đầu ra được khởi tạo lại,
phần backbone vẫn nạp nguyên. Chỉ cần thấy dòng `load pretrain successful` ngay sau đó là đúng.


In [ ]:
# 7) HUAN LUYEN — khoang 1-2 gio tren T4
#    MOC: sau moi 200 iter se in 'cur metric, acc: ...' — con so nay phai TANG DAN.
#    Giu tab mo, may khong sleep. Checkpoint luu MOI epoch nen dung giua chung van dung duoc.
DATA = '/content/data/rec_finetune'
!python tools/train.py -c {CONFIG} \
  -o Global.use_gpu=true \
     Global.pretrained_model=/content/pretrained/en_PP-OCRv5_mobile_rec_pretrained \
     Global.character_dict_path={DATA}/dict36.txt \
     Global.use_space_char=false \
     Global.max_text_length=10 \
     Global.epoch_num=30 \
     Global.save_epoch_step=1 \
     Global.eval_batch_step=[0,200] \
     Global.print_batch_step=20 \
     Global.save_model_dir=/content/output/rec_vn \
     Optimizer.lr.learning_rate=0.0001 \
     Optimizer.lr.warmup_epoch=1 \
     Train.dataset.data_dir={DATA} \
     Train.dataset.label_file_list=[{DATA}/train.txt] \
     Train.sampler.first_bs=128 \
     Train.loader.batch_size_per_card=128 \
     Train.loader.num_workers=2 \
     Eval.dataset.data_dir={DATA} \
     Eval.dataset.label_file_list=[{DATA}/val.txt] \
     Eval.loader.batch_size_per_card=128 \
     Eval.loader.num_workers=2


In [ ]:
# 7b) SAO LUU checkpoint len Drive — CHAY NGAY sau khi train xong (hoac khi dung giua chung)
#     Colab free hay ngat phien: mat /content la mat toan bo cong train.
!mkdir -p /content/drive/MyDrive/DATN/rec_vn_ckpt
!cp -f /content/output/rec_vn/best_accuracy.* /content/drive/MyDrive/DATN/rec_vn_ckpt/
!cp -f /content/output/rec_vn/latest.* /content/drive/MyDrive/DATN/rec_vn_ckpt/
!ls -la /content/drive/MyDrive/DATN/rec_vn_ckpt


In [ ]:
# 7c) TUY CHON — tiep tuc train tu checkpoint (khi Colab ngat giua chung).
#     Chay lai cell 1-6 truoc, roi bo dau # o duoi. Chi khac cell 7 o dong 'checkpoints'.
# !cp /content/drive/MyDrive/DATN/rec_vn_ckpt/latest.* /content/output/rec_vn/ 2>/dev/null
# DATA = '/content/data/rec_finetune'
# !python tools/train.py -c {CONFIG} \
#   -o Global.use_gpu=true Global.checkpoints=/content/output/rec_vn/latest \
#      Global.character_dict_path={DATA}/dict36.txt Global.use_space_char=false \
#      Global.max_text_length=10 Global.epoch_num=30 Global.save_epoch_step=1 \
#      Global.eval_batch_step=[0,200] Global.save_model_dir=/content/output/rec_vn \
#      Train.dataset.data_dir={DATA} Train.dataset.label_file_list=[{DATA}/train.txt] \
#      Train.sampler.first_bs=128 Train.loader.batch_size_per_card=128 \
#      Eval.dataset.data_dir={DATA} Eval.dataset.label_file_list=[{DATA}/val.txt]


In [ ]:
# 8) Danh gia checkpoint tot nhat tren tap val sach (571 mau, khong augment)
#    MOC: GHI LAI con so 'acc' — day la so mang ve de doi chieu voi baseline.
DATA = '/content/data/rec_finetune'
!python tools/eval.py -c {CONFIG} \
  -o Global.use_gpu=true \
     Global.checkpoints=/content/output/rec_vn/best_accuracy \
     Global.character_dict_path={DATA}/dict36.txt \
     Global.use_space_char=false \
     Global.max_text_length=10 \
     Eval.dataset.data_dir={DATA} \
     Eval.dataset.label_file_list=[{DATA}/val.txt] \
     Eval.loader.batch_size_per_card=128


In [ ]:
# 9) Xuat inference model va luu ve Drive
#    MOC: rec_vn_inference.zip xuat hien trong MyDrive/DATN/ (vai chuc MB).
DATA = '/content/data/rec_finetune'
!python tools/export_model.py -c {CONFIG} \
  -o Global.checkpoints=/content/output/rec_vn/best_accuracy \
     Global.character_dict_path={DATA}/dict36.txt \
     Global.use_space_char=false \
     Global.max_text_length=10 \
     Global.save_inference_dir=/content/output/rec_vn_inference
!ls -la /content/output/rec_vn_inference
!cd /content/output && zip -r -q rec_vn_inference.zip rec_vn_inference
!cp /content/output/rec_vn_inference.zip /content/drive/MyDrive/DATN/
!ls -la /content/drive/MyDrive/DATN/rec_vn_inference.zip


## Xong rồi thì làm gì tiếp

**1. Đưa model về máy**
Tải `rec_vn_inference.zip` từ Drive, giải nén vào `D:\DATN\models\rec_finetuned\`
(các file `inference.*` nằm **ngay trong** thư mục đó, không lồng thêm một cấp).

**2. Bật model mới** — mọi dây nối đã có sẵn, chỉ cần một biến môi trường:
```
# Docker, thêm vào .env:  ALPR_OCR_REC_MODEL_DIR=/app/models/rec_finetuned
# Chạy trực tiếp:         set ALPR_OCR_REC_MODEL_DIR=models/rec_finetuned
```
Đường dẫn sai sẽ **báo lỗi ngay lúc khởi động** thay vì âm thầm chạy model gốc.

**3. Đo lại trước khi tin** *(quan trọng nhất — xem `ai/training/README-rec-finetune.md`)*
- `ai/evaluation/ocr_accuracy.py` toàn tập, so với baseline trong `docs/reports/16-*`
- Ba bộ hồi quy: 16 ảnh lõi (`demo/images/expected.json`), 13 ca rescue, 3 video demo
- Ablation: chạy cả khi bật và khi tắt biến môi trường, để phần tăng quy đúng cho fine-tune

**Chỉ tiêu thành công đặt trước:** biển 2 dòng tăng **≥ 5 điểm** và biển 1 dòng **không giảm**.
Không đạt thì gỡ cờ, giữ model gốc, và ghi kết quả âm vào báo cáo — một thí nghiệm thất bại
có số liệu vẫn là nội dung tốt cho mục hạn chế của luận văn.
